# VortexTech Week 3: Regression and Clustering on Real Data
## AI & ML Internship Track

**Objective:** Tackle a realistic dataset using supervised regression and unsupervised clustering techniques.

**Dataset:** California Housing Dataset (20,640 samples, 8 features, continuous price target)

**Tasks:**
1. Build a regression model to predict median house prices
2. Apply K-Means clustering to find geographic/demographic groups
3. Evaluate both models with appropriate metrics

## Part 0: Setup and Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ All libraries imported successfully")

## Part 1: Data Loading and Exploration

### Load the California Housing Dataset

This dataset contains 20,640 records of California housing prices with 8 features:
- **MedInc**: Median income in block group
- **HouseAge**: Median house age in block group  
- **AveRooms**: Average number of rooms per household
- **AveBedrms**: Average number of bedrooms per household
- **Population**: Block group population
- **AveOccup**: Average occupancy per household
- **Latitude**: Block group latitude
- **Longitude**: Block group longitude
- **Target**: MedHouseVal (Median house value in $100,000s)

In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

# Create a complete dataframe
df = pd.concat([X, y], axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataset info:")
print(df.info())
print(f"\nBasic statistics:")
print(df.describe())

### Check for Missing Values

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)
print(f"\nTotal missing values: {missing_values.sum()}")
print("\n✓ No missing values detected - data is clean!")

### Visualize Feature Distributions

In [ ]:
# Plot distributions of key features
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(df.columns):
    axes[idx].hist(df[col], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Feature distribution plots saved")

### Correlation Analysis

In [ ]:
# Calculate correlation with target variable
correlation_with_target = df.corr()['MedHouseVal'].sort_values(ascending=False)
print("Correlation with Target (MedHouseVal):")
print(correlation_with_target)

# Heatmap of correlations
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            cbar_kws={'label': 'Correlation'}, square=True)
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Correlation analysis complete")

## Part 2: Regression Model - Predicting House Prices

### Decision: Random Forest vs Linear Regression

**Model Choice:** Random Forest Regressor

**Reasoning:**
- Linear Regression assumes linear relationship; Random Forest captures non-linear patterns
- Random Forest provides feature importance for interpretation
- Better generalization on real-world housing data
- Captures feature interactions (location × income)

### Step 1-2: Prepare Data and Train/Test Split

In [ ]:
# Separate features and target
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

### Step 3: Train Regression Models

In [ ]:
# Train Random Forest
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(
    n_estimators=100, max_depth=20, min_samples_split=5, random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("✓ Random Forest training complete")

# Train Linear Regression
print("\nTraining Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("✓ Linear Regression training complete")

### Step 4-5: Make Predictions and Evaluate

In [ ]:
# Predictions
y_pred_rf = rf_model.predict(X_test)
y_pred_lr = lr_model.predict(X_test)

# Evaluation function
def evaluate_model(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    print(f"\n{model_name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAE:  {mae:.4f}")
    return {'rmse': rmse, 'r2': r2, 'mae': mae}

print("="*50)
print("REGRESSION RESULTS")
print("="*50)
rf_metrics = evaluate_model(y_test, y_pred_rf, 'Random Forest')
lr_metrics = evaluate_model(y_test, y_pred_lr, 'Linear Regression')

### Step 6: Feature Importance

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance (Random Forest):")
print(feature_importance)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance Score', fontweight='bold')
plt.title('Random Forest Feature Importance', fontweight='bold', fontsize=14)
plt.gca().invert_yaxis()
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance visualization saved")

### Step 7: Visualize Predictions

In [ ]:
# Prediction comparison plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest
axes[0].scatter(y_test, y_pred_rf, alpha=0.5, s=20, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price ($100k)', fontweight='bold')
axes[0].set_ylabel('Predicted Price ($100k)', fontweight='bold')
axes[0].set_title(f'Random Forest (R² = {rf_metrics["r2"]:.4f})', fontweight='bold')
axes[0].grid(alpha=0.3)

# Linear Regression
axes[1].scatter(y_test, y_pred_lr, alpha=0.5, s=20, color='coral')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Price ($100k)', fontweight='bold')
axes[1].set_ylabel('Predicted Price ($100k)', fontweight='bold')
axes[1].set_title(f'Linear Regression (R² = {lr_metrics["r2"]:.4f})', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regression_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Prediction comparison plots saved")

## Part 3: Clustering - Finding Geographic and Demographic Groups

### Decision: Clustering Features

**Features:** Latitude, Longitude, Median Income

**Reasoning:**
- Geographic features capture regional patterns
- Income adds demographic dimension
- Captures real-world housing market segmentation

### Step 1-2: Prepare and Standardize Data

In [ ]:
# Select features for clustering
clustering_features = ['Latitude', 'Longitude', 'MedInc']
X_clustering = df[clustering_features].copy()

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering)

print(f"Clustering data prepared: {X_scaled.shape}")
print("✓ Features standardized (mean=0, std=1)")

### Step 3: Elbow Method - Determine Optimal Clusters

In [ ]:
# Elbow method
inertias = []
K_range = range(1, 11)

print("Running K-Means for k=1 to k=10...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    print(f"  k={k:2d}: Inertia = {kmeans.inertia_:.2f}")

print("\n✓ Elbow method calculation complete")

### Step 4: Plot Elbow Curve

In [ ]:
# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'o-', linewidth=2, markersize=8, color='steelblue')
plt.xlabel('Number of Clusters (k)', fontweight='bold', fontsize=12)
plt.ylabel('Inertia', fontweight='bold', fontsize=12)
plt.title('Elbow Method For Optimal k', fontweight='bold', fontsize=14)
plt.grid(alpha=0.3)
plt.xticks(K_range)
plt.axvline(x=4, color='red', linestyle='--', linewidth=2, label='Elbow at k=4')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Elbow curve saved")

### Step 5: Train Final K-Means Model

In [ ]:
# Final K-Means with k=4
optimal_k = 4
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)

print(f"Final K-Means Model (k={optimal_k}):")
print(f"  Inertia: {kmeans_final.inertia_:.2f}")
print(f"\nCluster distribution:")
for i in range(optimal_k):
    count = (cluster_labels == i).sum()
    pct = count / len(cluster_labels) * 100
    print(f"  Cluster {i}: {count:5d} samples ({pct:5.1f}%)")

### Step 6: Visualize Clusters

In [ ]:
# Geographic visualization
plt.figure(figsize=(12, 8))

colors = ['steelblue', 'coral', 'mediumseagreen', 'gold']
for i in range(optimal_k):
    mask = cluster_labels == i
    plt.scatter(df.loc[mask, 'Longitude'], df.loc[mask, 'Latitude'], 
               c=colors[i], label=f'Cluster {i}', s=30, alpha=0.6, edgecolors='black', linewidth=0.3)

# Plot cluster centers
centers = scaler.inverse_transform(kmeans_final.cluster_centers_)
plt.scatter(centers[:, 1], centers[:, 0], c='red', marker='X', s=400, 
           edgecolors='black', linewidth=2, label='Centers', zorder=5)

plt.xlabel('Longitude', fontweight='bold', fontsize=12)
plt.ylabel('Latitude', fontweight='bold', fontsize=12)
plt.title(f'K-Means Clustering (k={optimal_k}) - Geographic View', 
         fontweight='bold', fontsize=14)
plt.legend(loc='best', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('clusters_geographic.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Geographic cluster visualization saved")

### Step 7: Analyze Cluster Characteristics

In [ ]:
# Add cluster labels
df_clusters = df.copy()
df_clusters['Cluster'] = cluster_labels

print("\nCluster Characteristics:")
print("="*70)

for i in range(optimal_k):
    cluster_data = df_clusters[df_clusters['Cluster'] == i]
    print(f"\nCluster {i} ({len(cluster_data)} samples):")
    print("-" * 50)
    print(f"  Latitude:        {cluster_data['Latitude'].mean():.2f}")
    print(f"  Longitude:       {cluster_data['Longitude'].mean():.2f}")
    print(f"  Median Income:   ${cluster_data['MedInc'].mean():.2f}k")
    print(f"  Avg House Price: ${cluster_data['MedHouseVal'].mean()*100000:.0f}")
    print(f"  Avg House Age:   {cluster_data['HouseAge'].mean():.1f} years")

### Step 8: Income Analysis by Cluster

In [ ]:
# Income distribution by cluster
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Income vs Longitude
for i in range(optimal_k):
    mask = cluster_labels == i
    axes[0].scatter(df_clusters.loc[mask, 'Longitude'], 
                   df_clusters.loc[mask, 'MedInc'],
                   c=colors[i], label=f'Cluster {i}', s=30, alpha=0.6)

axes[0].set_xlabel('Longitude', fontweight='bold')
axes[0].set_ylabel('Median Income ($10k)', fontweight='bold')
axes[0].set_title('Clusters by Longitude vs Income', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Income distribution boxplot
cluster_income = [df_clusters[df_clusters['Cluster'] == i]['MedInc'].values for i in range(optimal_k)]
bp = axes[1].boxplot(cluster_income, labels=[f'C{i}' for i in range(optimal_k)],
                     patch_artist=True, notch=True)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

axes[1].set_ylabel('Median Income ($10k)', fontweight='bold')
axes[1].set_title('Income Distribution by Cluster', fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('clusters_income_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Income analysis visualization saved")

## Part 4: Final Summary

In [ ]:
print("\n" + "="*70)
print("WEEK 3 PROJECT SUMMARY")
print("="*70)
print(f"\n📊 REGRESSION:")
print(f"  Model: Random Forest (100 trees, depth=20)")
print(f"  Test R²: {rf_metrics['r2']:.4f} ({rf_metrics['r2']*100:.1f}% variance explained)")
print(f"  Test RMSE: ${rf_metrics['rmse']*100000:.0f}")
print(f"  Test MAE: ${rf_metrics['mae']*100000:.0f}")
print(f"\n🎯 CLUSTERING:")
print(f"  Algorithm: K-Means")
print(f"  Optimal Clusters: {optimal_k}")
print(f"  Features: Latitude, Longitude, Median Income")
print(f"  Elbow Point: Clear at k=4")
print(f"\n📁 Generated Files:")
print(f"  • feature_distributions.png")
print(f"  • correlation_heatmap.png")
print(f"  • feature_importance.png")
print(f"  • regression_comparison.png")
print(f"  • elbow_method.png")
print(f"  • clusters_geographic.png")
print(f"  • clusters_income_analysis.png")
print(f"\n✅ WEEK 3 PROJECT COMPLETE!")
print("="*70)